In [5]:
# =========================================
# 1. IMPORTS
# =========================================
import pandas as pd
import openpyxl
import re 

# =========================================
# 2. LOAD DATA
# =========================================
wb = openpyxl.load_workbook("downloads/RGB-T_screened.xlsx")
ws = wb.active

data = list(ws.values)

# récupérer header + données
df = pd.DataFrame(data[1:], columns=data[0])

print("Records loaded:", len(df))


# =========================================
# 3. NORMALIZATION
# =========================================
def norm(text):
    return re.sub(r"\s+", " ", str(text).lower()).strip()


# =========================================
# 4. KEYWORDS (PRISMA CRITERIA)
# =========================================

# ❌ Non original
NON_ORIGINAL = [
    "review", "survey", "meta-analysis",
    "systematic review", "editorial",
    "commentary", "letter", "tutorial"
]

# ❌ No method
NO_METHOD = [
    "analysis of", "discussion", "overview"
]

# ❌ Not RGB-T central
NOT_RGBT = [
    "rgb only", "thermal only", "infrared only"
]

# ❌ Out of scope tasks
OUT_SCOPE = [
    "classification", "reconstruction",
    "enhancement", "compression"
]

# ❌ Irrelevant domain
IRRELEVANT = [
    "medical", "disease", "health",
    "finance", "marketing"
]


# =========================================
# 5. HELPER FUNCTION
# =========================================
def contains_any(text, keywords):
    return any(k in text for k in keywords)


# =========================================
# 6. EXCLUSION FUNCTION (PRISMA)
# =========================================
def exclusion_criteria(title, abstract):

    text = norm(title) + " " + norm(abstract)

    # 1. Non-original
    if contains_any(text, NON_ORIGINAL):
        return "Exclude", "Non-original study"

    # 2. No method
    if contains_any(text, NO_METHOD):
        return "Exclude", "No method/model proposed"

    # 3. Not RGB-T central
    if contains_any(text, NOT_RGBT):
        return "Exclude", "RGB-T not central"

    # 4. Out of scope
    if contains_any(text, OUT_SCOPE):
        return "Exclude", "Task out of scope"

    # 5. Irrelevant domain
    if contains_any(text, IRRELEVANT):
        return "Exclude", "Irrelevant domain"

    # ✅ Otherwise keep
    return "Include", "Relevant"


# =========================================
# 7. APPLY FILTER
# =========================================

if "Abstract" not in df.columns:
    df["Abstract"] = ""

df[["Decision", "Reason"]] = df.apply(
    lambda row: pd.Series(
        exclusion_criteria(row["Title"], row["Abstract"])
    ),
    axis=1
)

# =========================================
# 8. SUMMARY
# =========================================
print("\nDecision counts:")
print(df["Decision"].value_counts())


# =========================================
# 9. SAVE RESULTS
# =========================================
df.to_excel("downloads/RGB-T_Verified_screened.xlsx", index=False)

print("\nSaved: downloads/RGB-T_Verified_screened.xlsx")

Records loaded: 135

Decision counts:
Include    129
Exclude      6
Name: Decision, dtype: int64

Saved: downloads/RGB-T_Verified_screened.xlsx
